# Entropy & KL Divergence

Companion notebook for the [Entropy & KL Divergence](https://ml-viz.vercel.app/courses/probability-statistics/05-entropy-and-kl-divergence) lesson on ML Viz.

We'll compute surprise, entropy, cross-entropy, and KL divergence numerically, verify the identity H(p,q) = H(p) + KL(p‖q), and watch the two KL directions behave differently on a bimodal target.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.facecolor"] = "#0f1117"
plt.rcParams["axes.facecolor"] = "#1a1d27"
plt.rcParams["axes.edgecolor"] = "#2e3347"
plt.rcParams["grid.color"] = "#2e3347"

def entropy(p):
    p = p[p > 0]
    return -(p * np.log(p)).sum()

def cross_entropy(p, q):
    return -(p[p > 0] * np.log(q[p > 0])).sum()

def kl(p, q):
    mask = p > 0
    return (p[mask] * np.log(p[mask] / q[mask])).sum()

## 1. Entropy across coin biases

Entropy peaks at the fair coin and collapses toward certainty at the edges.

In [ ]:
thetas = np.linspace(0.001, 0.999, 200)
H = [entropy(np.array([t, 1 - t])) for t in thetas]

plt.figure(figsize=(8, 3.5))
plt.plot(thetas, H, color="#6366f1", lw=2)
plt.axvline(0.5, color="#eab308", ls="--", label="fair coin: max entropy")
plt.xlabel("P(heads)")
plt.ylabel("entropy (nats)")
plt.title("Uncertainty is maximal when you know least")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

## 2. The identity H(p, q) = H(p) + KL(p‖q)

Cross-entropy = irreducible uncertainty of the data + your modeling penalty.

In [ ]:
p = np.array([0.7, 0.2, 0.1])
q = np.array([0.5, 0.3, 0.2])

print(f"H(p)      = {entropy(p):.4f}")
print(f"H(p, q)   = {cross_entropy(p, q):.4f}")
print(f"KL(p‖q)   = {kl(p, q):.4f}")
print(f"H + KL    = {entropy(p) + kl(p, q):.4f}   (matches cross-entropy)")
print()
print(f"KL(p‖q) = {kl(p, q):.4f}  vs  KL(q‖p) = {kl(q, p):.4f}   (asymmetric!)")

## 3. Cross-entropy IS the classification loss

A one-hot data distribution collapses cross-entropy to −log q(true class) — the familiar log-loss.

In [ ]:
q_pred = np.array([0.05, 0.85, 0.10])      # model's softmax output
p_true = np.array([0.0, 1.0, 0.0])           # one-hot truth (class 1)

print(f"cross-entropy      : {cross_entropy(p_true, q_pred):.4f}")
print(f"-log q(true class) : {-np.log(q_pred[1]):.4f}")

## 4. Forward vs reverse KL on a bimodal target

Fit a single Gaussian q to a two-bump p by minimizing each direction (crude grid search). Forward KL must cover both bumps; reverse KL happily picks one.

In [ ]:
x = np.linspace(-6, 6, 601)
dx = x[1] - x[0]

def gauss(x, mu, sd):
    g = np.exp(-((x - mu) ** 2) / (2 * sd**2))
    return g / (g.sum() * dx)

p_bimodal = 0.5 * gauss(x, -2, 0.6) + 0.5 * gauss(x, 2, 0.6)
p_disc = p_bimodal * dx                       # discretize to probabilities

best = {}
for direction in ["forward", "reverse"]:
    best_kl, best_q = np.inf, None
    for mu in np.linspace(-3, 3, 61):
        for sd in np.linspace(0.4, 3.5, 32):
            q_disc = gauss(x, mu, sd) * dx
            d = kl(p_disc, q_disc) if direction == "forward" else kl(q_disc, p_disc)
            if d < best_kl:
                best_kl, best_q = d, q_disc
    best[direction] = best_q

plt.figure(figsize=(9, 4))
plt.plot(x, p_disc / dx, color="#94a3b8", lw=2, label="target p (bimodal)")
plt.plot(x, best["forward"] / dx, color="#14b8a6", lw=2, label="argmin KL(p‖q): covers both modes")
plt.plot(x, best["reverse"] / dx, color="#f43f5e", lw=2, label="argmin KL(q‖p): picks one mode")
plt.xlabel("x")
plt.ylabel("density")
plt.title("Same formula, opposite personalities")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

**Next:** see these losses in the wild — the [course quiz](https://ml-viz.vercel.app/courses/probability-statistics/06-quiz), or jump to [VAEs](https://ml-viz.vercel.app/courses/generative-models/03-variational-autoencoders) where the reverse-KL story plays out in the ELBO.